# 02. Pipeline RAG passo a passo

Este notebook abre a caixa: segue **uma pergunta real** por cada etapa do pipeline e mostra o
que entra e o que sai de cada uma, exatamente com o código de produção (`src/voc_rag`).

| Etapa do enunciado | Onde está no código |
|---|---|
| Transformação da pergunta | `retrieval/query.py` |
| Representação vetorial / embeddings | `indexing/embeddings.py` |
| Recuperação semântica (e lexical) | `retrieval/retriever.py`, `indexing/vector_store.py`, `indexing/bm25_index.py` |
| Seleção dos conteúdos mais relevantes | `retrieval/rerank.py`, `retrieval/retriever.py` (limiar, deduplicação, MMR) |
| Construção do contexto | `generation/answer.py` |
| Geração da resposta | `generation/llm.py`, `generation/answer.py`, `pipeline.py` |
| Abstenção | `pipeline.py` (três camadas) |

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json
import numpy as np
import pandas as pd

from voc_rag.generation.llm import LLMClient
from voc_rag.retrieval.query import analisa_pergunta
from voc_rag.retrieval.retriever import (RetrievalConfig, consultas_de, filtros_efetivos,
                                         get_kb, recupera, rrf)
from voc_rag.generation.answer import (SYSTEM, RespostaGerada, monta_contexto, monta_prompt,
                                       rotula_evidencias, valida_citacoes)
from voc_rag.pipeline import VoCPipeline, render_markdown

pd.set_option("display.max_colwidth", 110)
kb = get_kb(); kb.aquece()
llm = LLMClient()
cfg = RetrievalConfig()
PERGUNTA = "Quais reclamações relacionadas à entrega aparecem com maior frequência nas avaliações com nota baixa?"
print(f"Base: {len(kb.docs):,} documentos | embedding: {kb.modelo} ({kb.emb.shape[1]} dimensões)")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Base: 42,138 documentos | embedding: BAAI/bge-m3 (1024 dimensões)


## Etapa 1. Transformação da pergunta

Uma chamada ao LLM (saída JSON validada por Pydantic) converte a pergunta do gestor em
instruções de busca. O ponto central: a pergunta está em linguagem **analítica**
("quais reclamações aparecem com maior frequência"), os documentos em linguagem de
**cliente** ("comprei dia 10 e até hoje nada"). A reescrita aproxima os dois registros.

In [2]:
analise = analisa_pergunta(PERGUNTA, kb.categorias, llm)
print(json.dumps(analise.model_dump(exclude={"origem"}), ensure_ascii=False, indent=2))

{
  "experiencia_de_compra": true,
  "posterior_a_2018": false,
  "tenta_manipular": false,
  "escopo": "dentro",
  "motivo_escopo": "A pergunta aborda a experiência de compra dos clientes relacionada a entregas e avaliações com nota baixa, tema totalmente contido na base de avaliações.",
  "tipo": "panorama",
  "polaridade": "negativa",
  "filtros": {
    "nota_min": 1,
    "nota_max": 2,
    "sentimentos": [],
    "categorias": [],
    "entrega_status": [],
    "ano_mes_inicio": null,
    "ano_mes_fim": null,
    "ufs": [],
    "regioes": [],
    "pagamentos": [],
    "valor_min": null,
    "valor_max": null,
    "multi_vendedor": null
  },
  "consulta_busca": "atraso na entrega produto nao chegou prazo nao cumprido mercadoria extraviada",
  "consultas_alternativas": [
    "entrega atrasada demorou muito para chegar",
    "nao recebi meu pedido rastreio nao atualiza"
  ],
  "avaliacao_hipotetica": "O produto passou muito do prazo de entrega e até agora não chegou nada, descaso total 

In [3]:
f = filtros_efetivos(analise, cfg)
print("Filtros extraídos da pergunta:", analise.filtros.descricao())
print("Filtros efetivos na busca    :", f.descricao())
print(f"Documentos no recorte        : {f.mask(kb.docs).sum():,} de {len(kb.docs):,}")

Filtros extraídos da pergunta: nota de 1 a 2
Filtros efetivos na busca    : nota de 1 a 2
Documentos no recorte        : 10,845 de 42,138


**Leitura.** "nota baixa" virou `nota_max = 2` (filtro de metadado aplicado dentro do banco
vetorial), o tipo `panorama` vai ampliar o contexto e ligar o quadro de frequências, e a
consulta reescrita usa o vocabulário dos clientes. O LLM também propõe variações (multi-query)
e uma avaliação hipotética (HyDE); as duas foram medidas no notebook 03, **não trouxeram
ganho** (empate técnico; puxam temas vizinhos e custam chamadas) e ficam desligadas no sistema final. Elas aparecem abaixo
só para comparação.

## Etapa 2. Representação vetorial

Pergunta e documentos passam pelo mesmo modelo de embedding (`BAAI/bge-m3`, multilíngue,
1024 dimensões, vetores normalizados). A similaridade é o cosseno. O teste abaixo mede quanto
a reescrita aproxima a consulta de avaliações que de fato falam de problema de entrega.

In [4]:
consultas = consultas_de(analise, cfg, PERGUNTA)   # sistema final: só a consulta reescrita
exemplos = kb.docs[kb.docs["texto"].str.contains("não recebi|atras", case=False, regex=True)
                   & (kb.docs["nota"] <= 2)].sample(200, random_state=42)
vd = kb.emb[kb.docs_pos.loc[exemplos["review_id"]].to_numpy()]
textos = {"pergunta crua": PERGUNTA, "consulta reescrita (usada)": analise.consulta_busca,
          **{f"variação {i+1} (desligada)": q for i, q in enumerate(analise.consultas_alternativas)},
          "avaliação hipotética, HyDE (desligada)": analise.avaliacao_hipotetica}
vq = kb.embedder.encode_queries(list(textos.values()))
sims = pd.DataFrame({
    "texto da consulta": list(textos.values()),
    "cosseno médio com 200 reclamações de entrega": (vq @ vd.T).mean(axis=1).round(3),
}, index=list(textos))
sims

,texto da consulta,cosseno médio com 200 reclamações de entrega
pergunta crua,Quais reclamações relacionadas à entrega aparecem com maior frequência nas avaliações com nota baixa?,0.514
consulta reescrita (usada),atraso na entrega produto nao chegou prazo nao cumprido mercadoria extraviada,0.647
variação 1 (desligada),entrega atrasada demorou muito para chegar,0.664
variação 2 (desligada),nao recebi meu pedido rastreio nao atualiza,0.644
"avaliação hipotética, HyDE (desligada)","O produto passou muito do prazo de entrega e até agora não chegou nada, descaso total com o consumidor.",0.684


## Etapa 3. Recuperação semântica e lexical

Cada consulta vai a dois índices:

* **denso** (ChromaDB, cosseno, filtro de metadado no `where`): captura sinônimos e paráfrases;
* **BM25** (stemming em português, negações preservadas): captura termos exatos
  ("correios", "nota fiscal") que o vetor às vezes dilui.

As listas são fundidas por **RRF** (Reciprocal Rank Fusion), que usa só a posição de cada
documento em cada lista: cosseno e BM25 não estão na mesma escala, a posição está. **A ordem
da fusão é a ordem final** (etapa 4 explica por quê).

In [5]:
where = f.to_chroma_where()
mask = f.mask(kb.docs)
qv = kb.embedder.encode_queries(consultas)
densas = kb.store.query(qv, k=5, where=where)
lex = kb.bm25.search(consultas[0], k=5, mask=mask)

def mostra(lista, rotulo):
    return pd.DataFrame([{"lista": rotulo, "pos": i + 1, "score": round(s, 3),
                          "texto": kb.docs.loc[d, "texto"]} for i, (d, s) in enumerate(lista)])

pd.concat([mostra(densas[0], "denso, consulta reescrita"), mostra(lex, "BM25, consulta reescrita")])

,lista,pos,score,texto
0,"denso, consulta reescrita",1,0.889,"atraso na entrega da mercadoria,não cumpre prazo determinado."
1,"denso, consulta reescrita",2,0.862,entrega atrasada. o prazo terminou e nao recebi meu produto
2,"denso, consulta reescrita",3,0.859,Atraso. Produto não chegou no prazo
3,"denso, consulta reescrita",4,0.844,meu produto nao chegou esta atrasado.
4,"denso, consulta reescrita",5,0.842,Produto com entrega atrasada
0,"BM25, consulta reescrita",1,8.019,"atraso na entrega da mercadoria,não cumpre prazo determinado."
1,"BM25, consulta reescrita",2,6.824,não cumpre prazo. Não cumpriram prazo de entrega e nem avisaram o atraso.
2,"BM25, consulta reescrita",3,6.470,mercadoria não entregue. Mercadoria em atraso. xxxxxxxxxxxxx
3,"BM25, consulta reescrita",4,5.837,Não cumpriu com o prazo e o produto não chegou.
4,"BM25, consulta reescrita",5,5.733,Não recebi a mercadoria...ESTÁ MUITO ATRASADA A ENTREGA...


In [6]:
res = recupera(PERGUNTA, analise, cfg, kb)
c = res.candidatos
print(f"Listas fundidas: {len(consultas)} densa(s) + {len(consultas)} lexical(is)")
print(f"Candidatas após RRF (top {cfg.n_rerank}): {len(c)}")
print(f"Recuperadas pelos dois índices: {(c['score_denso'].notna() & c['score_bm25'].notna()).sum()}"
      f" | só denso: {(c['score_denso'].notna() & c['score_bm25'].isna()).sum()}"
      f" | só BM25: {(c['score_denso'].isna() & c['score_bm25'].notna()).sum()}")
c.sort_values("pos_rrf")[["pos_rrf", "rrf", "score_denso", "score_bm25", "nota", "texto"]].head(8)

Listas fundidas: 1 densa(s) + 1 lexical(is)
Candidatas após RRF (top 80): 80
Recuperadas pelos dois índices: 8 | só denso: 36 | só BM25: 36


,pos_rrf,rrf,score_denso,score_bm25,nota,texto
0,1,0.032787,0.888661,8.018528,1,"atraso na entrega da mercadoria,não cumpre prazo determinado."
1,2,0.030798,0.859103,5.530151,2,Atraso. Produto não chegou no prazo
2,3,0.029710,0.826100,5.837031,1,Não cumpriu com o prazo e o produto não chegou.
3,4,0.028446,0.844138,5.122379,1,meu produto nao chegou esta atrasado.
4,5,0.026129,0.861880,4.604811,1,entrega atrasada. o prazo terminou e nao recebi meu produto
5,6,0.025307,0.801586,6.470161,1,mercadoria não entregue. Mercadoria em atraso. xxxxxxxxxxxxx
6,7,0.024585,0.839922,4.481288,1,Atraso entrega. Não recebi o produto
7,8,0.022727,0.808530,4.766141,1,Produto não chegou está com 2 dias de atraso


## Etapa 4. Seleção dos conteúdos mais relevantes

1. **Reranker cross-encoder como porteiro** (`BAAI/bge-reranker-v2-m3`): lê o par (consulta,
   avaliação) junto e devolve relevância em [0, 1]. Pontuamos contra a consulta reescrita, não
   contra a pergunta analítica (a célula abaixo mostra por quê). Ele **não reordena**: no
   notebook 03, ordenar pelo reranker piorou a precisão, porque ele confunde temas vizinhos
   ("produto veio quebrado" no topo de uma pergunta sobre embalagem). Ele decide quem é
   relevante o bastante para entrar e quando não há evidência suficiente.
2. **Limiar 0,30** e **mínimo de 3 evidências** (escolhidos no notebook 03).
3. **Deduplicação**: "muito bom" de 50 clientes entra uma vez, com a contagem anotada.
4. **MMR**: entre as relevantes, prefere as que acrescentam algo diferente, mantendo a ordem da
   fusão.

In [7]:
textos_c = c["texto"].tolist()
contra_pergunta = kb.reranker.score(PERGUNTA, textos_c)
comparacao = pd.DataFrame({
    "reranker contra a pergunta analítica": np.quantile(contra_pergunta, [.25, .5, .75]).round(3),
    "reranker contra a consulta reescrita": np.quantile(c["relevancia"], [.25, .5, .75]).round(3),
}, index=["quartil 1", "mediana", "quartil 3"])
comparacao

,reranker contra a pergunta analítica,reranker contra a consulta reescrita
quartil 1,0.0,0.870
mediana,0.0,0.937
quartil 3,0.0,0.977


In [8]:
print(f"Candidatas: {len(c)} | acima do limiar ({cfg.limiar_relevancia}): {res.n_relevantes} | "
      f"no contexto após deduplicação e MMR: {len(res.selecionados)}")
res.selecionados[["pos_rrf", "relevancia", "repeticoes_no_topo", "nota", "entrega_status", "texto"]].head(12)

Candidatas: 80 | acima do limiar (0.3): 79 | no contexto após deduplicação e MMR: 30


,pos_rrf,relevancia,repeticoes_no_topo,nota,entrega_status,texto
0,1,0.997007,1,1,atrasado,"atraso na entrega da mercadoria,não cumpre prazo determinado."
1,2,0.998522,1,2,atrasado,Atraso. Produto não chegou no prazo
2,3,0.990025,1,1,nao_entregue,Não cumpriu com o prazo e o produto não chegou.
3,4,0.991748,1,1,atrasado,meu produto nao chegou esta atrasado.
4,5,0.954223,1,1,atrasado,entrega atrasada. o prazo terminou e nao recebi meu produto
5,6,0.949763,1,1,atrasado,mercadoria não entregue. Mercadoria em atraso. xxxxxxxxxxxxx
6,7,0.931088,1,1,atrasado,Atraso entrega. Não recebi o produto
7,8,0.982288,1,1,atrasado,Produto não chegou está com 2 dias de atraso
8,9,0.870707,1,1,atrasado,não cumpre prazo. Não cumpriram prazo de entrega e nem avisaram o atraso.
9,11,0.697197,1,1,atrasado,Não recebi a mercadoria...ESTÁ MUITO ATRASADA A ENTREGA...


## Etapa 5. Construção do contexto

As evidências viram uma lista numerada `[R1]..[Rn]`. Cada uma leva os metadados que ajudam o
modelo a interpretar o texto (nota, situação e atraso da entrega, categoria, UF, mês). Em
perguntas de panorama e de evidência entra também o **quadro de frequências**: contagem de aspectos na base
completa do recorte, para o modelo não confundir a amostra recuperada com a base inteira.

In [9]:
from voc_rag.themes.taxonomy import quadro_frequencias
evid = rotula_evidencias(res.selecionados)
quadro_df, quadro_txt = quadro_frequencias(kb.docs, res.filtros_aplicados, analise.polaridade) \
    if analise.tipo == "panorama" else (None, None)
prompt = monta_prompt(PERGUNTA, monta_contexto(evid), res.filtros_aplicados.descricao(),
                      res.n_universo, quadro_txt)
print(prompt[:3500], "\n[...]")

PERGUNTA DO GESTOR: Quais reclamações relacionadas à entrega aparecem com maior frequência nas avaliações com nota baixa?

RECORTE DA BASE: nota de 1 a 2 (10845 avaliações com comentário no recorte)

QUADRO DE FREQUÊNCIAS (base completa do recorte):
- Produto não recebido: ~4569 avaliações (42,1% das 10845 do recorte)
- Atendimento ruim: ~2800 avaliações (25,8% das 10845 do recorte)
- Atraso na entrega: ~2641 avaliações (24,4% das 10845 do recorte)
- Produto diferente do anunciado: ~2607 avaliações (24,0% das 10845 do recorte)
- Troca, devolução ou reembolso: ~2046 avaliações (18,9% das 10845 do recorte)
- Pedido incompleto: ~1906 avaliações (17,6% das 10845 do recorte)
- Defeito ou avaria: ~1540 avaliações (14,2% das 10845 do recorte)
- Qualidade abaixo do esperado: ~1438 avaliações (13,3% das 10845 do recorte)
- Embalagem ruim: ~192 avaliações (1,8% das 10845 do recorte)
(Estimativa de um classificador de aspectos com correção de erro; uma avaliação pode ter mais de um aspecto. Use c

 
[...]


## Etapa 6. Geração e validação

O LLM recebe as regras de fundamentação (prompt de sistema abaixo) e devolve JSON com resumo,
insights e os ids das evidências de cada insight. O código então **valida cada citação**
contra o contexto: id inexistente é removido e insight sem evidência é descartado.

In [10]:
print(SYSTEM)

Você é analista de Voice of Customer de um e-commerce. Responda à pergunta do gestor
usando EXCLUSIVAMENTE as avaliações de clientes fornecidas como evidências [R1]..[Rn] e,
quando houver, o QUADRO DE FREQUÊNCIAS.

Regras:
1. Toda afirmação sobre o que clientes relatam cita evidências no texto, uma marca por id:
   [R3][R7]. No texto, cite no máximo 4 evidências por frase (as mais representativas); em
   "evidencias", liste TODAS as que sustentam o insight (isso permite contar o apoio).
   Nunca cite um id que não exista.
2. Não use conhecimento externo. Não invente causas, números, percentuais, produtos,
   vendedores ou datas que não estejam nas evidências.
3. Números só podem vir (a) do QUADRO DE FREQUÊNCIAS, dizendo que se referem à base
   completa do recorte, ou (b) de contagens das próprias evidências ("4 das 15 avaliações
   recuperadas"). Nunca extrapole a amostra recuperada para a base inteira. Um número do
   quadro fica numa frase SEM marcas [R#] (as evidências não o compro

In [11]:
gerada = llm.generate(SYSTEM, prompt, RespostaGerada, etapa="geracao", pensamento="alto")
validada, relatorio = valida_citacoes(gerada, set(evid["rotulo"]))
print("Relatório de citações:", vars(relatorio))

Relatório de citações: {'citacoes_total': 19, 'citacoes_invalidas': 0, 'insights_total': 3, 'insights_descartados': 0, 'ids_invalidos': []}


In [12]:
from IPython.display import Markdown
Markdown(render_markdown(validada, len(evid)))

Os clientes com avaliações baixas relatam majoritariamente problemas graves com o descumprimento de prazos e a falta de recebimento dos produtos [R1][R3][R6][R13]. Adicionalmente, há críticas severas quanto à ausência de avisos sobre os atrasos e à dificuldade de rastreamento das encomendas [R9][R13][R23].

- **Atraso excessivo nos prazos**: Os consumidores apontam com alta frequência que a loja não cumpre as datas estabelecidas, resultando em longos períodos de espera após o prazo limite [R1][R2][R4][R5]. *(apoio: 17 de 30 evidências)*
- **Ausência de recebimento do produto**: Diversos relatos indicam que a mercadoria não chegou ao destino final ou que os pedidos foram cancelados sem resolução [R3][R6][R17][R25]. O indicador de produto não recebido representa 42,1% das ocorrências na base completa do recorte. *(apoio: 12 de 30 evidências)*
- **Falta de comunicação e rastreio**: Os compradores criticam a ausência de informações preventivas sobre os atrasos e a dificuldade para localizar o paradeiro do pacote [R9][R13][R20][R23]. *(apoio: 4 de 30 evidências)*

*Limitações: As análises detalhadas baseiam-se em uma amostra recuperada por relevância, e os percentuais numéricos refletem a base completa do recorte.*

In [13]:
citadas = sorted({e for i in validada.insights for e in i.evidencias}, key=lambda s: int(s[1:]))
evid[evid["rotulo"].isin(citadas)][["rotulo", "review_id", "nota", "texto"]].head(10)

,rotulo,review_id,nota,texto
0,R1,10467dc456e818c60bd750271b5d183e,1,"atraso na entrega da mercadoria,não cumpre prazo determinado."
1,R2,48b3ce236ea9d35a4850ea94ca9ca38f,2,Atraso. Produto não chegou no prazo
2,R3,a4f9f285b2ed89a616d93dea024eb31b,1,Não cumpriu com o prazo e o produto não chegou.
3,R4,4a7118b1f9c3d06869e0cd9d5c1b4ccb,1,meu produto nao chegou esta atrasado.
4,R5,1bfec3e913196c72c1ea06fc8816bd9c,1,entrega atrasada. o prazo terminou e nao recebi meu produto
5,R6,0985210ad6123c6554ec19639a43cea2,1,mercadoria não entregue. Mercadoria em atraso. xxxxxxxxxxxxx
6,R7,9d6c1f490b2b4bfef33820b5679e2d82,1,Atraso entrega. Não recebi o produto
7,R8,b28bfbd00dfd0c6d80e9cb524e0f205e,1,Produto não chegou está com 2 dias de atraso
8,R9,85a391b84d8e864de7190a2afb6f82e4,1,não cumpre prazo. Não cumpriram prazo de entrega e nem avisaram o atraso.
9,R10,1a6763b35ea712ac4530a70a61898a6c,1,Não recebi a mercadoria...ESTÁ MUITO ATRASADA A ENTREGA...


**Rastreabilidade.** Cada `[R#]` da resposta leva a um `review_id` real da base da Olist; o
gestor pode auditar qualquer insight lendo as avaliações que o sustentam.

## Etapa 7. Quando não responder

Três camadas, da mais barata para a mais cara:

1. **Escopo** (análise da pergunta): pergunta fora da base é recusada antes de qualquer busca;
2. **Relevância** (reranker): nenhuma avaliação acima do limiar, ou menos que o mínimo de
   evidências, encerra sem chamar o gerador;
3. **Suficiência** (geração): o LLM declara `suficiente=false` quando as evidências tocam o
   tema mas não respondem (ex.: falam de entrega, mas nenhuma de entrega por drone).

In [14]:
pipe = VoCPipeline(llm=llm, kb=kb, aquecer=False)
for p in ["Qual foi o faturamento da Olist em 2018?",
          "O que os clientes dizem sobre a instalação de ar-condicionado feita pela loja?",
          "O que os clientes acharam das entregas feitas por drone?"]:
    r = pipe.responder(p)
    print(f"\n>>> {p}\nstatus: {r.status}\n{r.markdown[:400]}")


>>> Qual foi o faturamento da Olist em 2018?
status: fora_escopo
**Fora do escopo da base de conhecimento.**

A base contém apenas avaliações escritas por clientes da Olist entre 2016 e 2018 sobre suas compras (entrega, produto, atendimento, vendedor, embalagem, preço). Não respondo com conhecimento externo para não apresentar algo que os dados não sustentam.



>>> O que os clientes dizem sobre a instalação de ar-condicionado feita pela loja?
status: sem_evidencia
**Não encontrei avaliações relevantes para responder com segurança.** Analisei 58 candidatas no recorte (categoria climatizacao; 104 avaliações com comentário), e nenhuma atingiu a relevância mínima (0,30). Prefiro não responder a inventar uma conclusão sem suporte nos dados.



>>> O que os clientes acharam das entregas feitas por drone?
status: nao_respondida
**As evidências recuperadas não sustentam uma resposta.** As evidências recuperadas tratam apenas de prazos e rapidez geral de entrega, sem conter informações sobre entregas feitas por drone [R1][R3][R6][R9]. Portanto, os dados disponíveis não permitem responder à pergunta do gestor.
